In [ ]:
import warnings
from transformers import logging

# Hide all transformers warnings
logging.set_verbosity_error()

# Hide python warnings
warnings.filterwarnings("ignore")


In [ ]:
# Install Transformer library
!pip install transformers torch

In [ ]:
#importing mmodel using transformer
from transformers import pipeline

In [ ]:
chatbot = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.1",
    pad_token_id=2,
    max_new_tokens=300
)

In [ ]:
#prompt engineering to make the responses friendly and clear
SYSTEM_PROMPT = """
You are a helpful medical assistant chatbot. Act like a top 1% helpful medical assistants. Give simple and friendly health information.

Safety Rules:
- Do not diagnose diseases.
- Do not prescribe medicines.
- Do not give exact dosage or overdose limits.
- For medicine questions, do not simply say "yes". Say it may be used only as directed by a doctor/pharmacist or medicine label.
- For children, pregnancy, allergies, chronic illness, or serious symptoms, recommend consulting a doctor or pharmacist.
"""

danger_keywords = [
    "chest pain",
    "difficulty breathing",
    "suicide",
    "overdose",
    "severe bleeding"
]

In [ ]:
#create safety filter function
def safety_filter(query):
    query = query.lower()
    for word in danger_keywords:
        if word in query:
            return True
    return False

In [ ]:
def health_chatbot(user_query):

    if safety_filter(user_query):
        return "This may be serious. Please contact a doctor or emergency service immediately."

    prompt = f"""
    {SYSTEM_PROMPT}

    User: {user_query}

    Assistant:
    """

    response = chatbot(
        prompt,
        temperature=0.5,
        do_sample=True,
        return_full_text=False,
        eos_token_id=2
    )

    answer = response[0]["generated_text"].strip()

    # Remove extra generated chat
    stop_words = ["User:", "Assistant:"]

    for word in stop_words:
        if word in answer:
            answer = answer.split(word)[0]

    return answer.strip()

In [ ]:
# Create chatbot function
def health_chatbot(user_query):
  if safety_filter(user_query):
        return "This may be a serious medical emergency. Please contact a doctor or emergency service immediately."

  prompt = SYSTEM_PROMPT + "\nUser: " + user_query + "\nAssistant:"

  response = chatbot(
        prompt,
        do_sample=True,
        temperature=0.7,
        return_full_text=False
    )

  answer = response[0]["generated_text"].strip()

  return answer


In [ ]:
# Test Questions
questions = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?"
]

for q in questions:

    print("\nUser:", q)
    print("Bot:", health_chatbot(q))
    print("-" * 50)

In [ ]:
#to take real time input and test model
print("Health Assistant Chatbot")
print("Type your health question below. Type 'quit' to exit.")

user_query = ""

while user_query != "q":

    user_query = input("\n Hey Wassup please let us know how i can assist you: ")

    if user_query == "q":
        print("Thanks for choosing General Health Query Chatbot")
        break

    response = health_chatbot(user_query)
    print("\n \n Bot:", response)
